In [1]:
%matplotlib inline
import torch
from d2l import torch as d2l

In [6]:
class LinearRegression(d2l.Module):
    """Linear Regression Model"""
    def __init__(self, num_inputs, lr, sigma=0.01):
        super().__init__()
        self.save_hyperparameters()
        self.w = torch.normal(0, sigma, (num_inputs, 1), requires_grad = True) #initliaze weights
        self.b = torch.zeros(1, requires_grad = True) #initiialize bias

In [ ]:
@d2l.add_to_class(LinearRegression)
def forward(self, X):
    return torch.matmul(X, self.w) + self.b #compute the y_hat = Xw + b function for every row of X

In [7]:
@d2l.add_to_class(LinearRegression)
def loss(self, y_hat , y):
    l = (y_hat - y) ** 2 // 2 #sqaured loss function 
    return l.mean()

In [8]:
class SGD(d2l.HyperParameters):
    """Minibatch Stochastic Gradient Descent"""
    def __init__(self, params, lr):
        self.save_hyperparameters()

    def step(self):
        for param in self.params:
            param -= self.lr * param.grad #subtract learning rate * gradient from paramter because gradient points in the direction that increases loss, so to reduce loss subtract

    def zero_grad(self):
        for param in self.params:
            if (param.grad is not None):
                param.grad_zero(); #reset each batch to avoid carry overs
            
@d2l.add_to_class(LinearRegression) 
def configure_optimizers(self):
    return SGD([self.w, self.b], self.lr) #instance of SGD class

In [9]:
#training: initialize paramters, for each minbatch -> compute loss, compute gradients, update params
@d2l.add_to_class(d2l.Trainer)  
def prepare_batch(self, batch):
    return batch

@d2l.add_to_class(d2l.Trainer)
def fit_epoch(self):
    self.model.train() #put model in training mode
    for batch in self.train_dataloader: #one pass through dataset
        loss = self.model.training_step(self.prepare_batch(batch)) #yhat function and loss funciton 
        self.optim.zero_grad()
        with torch.no_grad():
            loss.backward() #compute gradient
            if self.gradient_clip_val > 0:#caps gradient size
                self.clip_gradients(self.gradient_clip_val, self.model)
            self.optim.step() #update step
        self.train_batch_idx += 1
    if self.val_dataloader is None: #learn from predicitons basically
        return
    self.model.eval() #eval mode
    for batch in self.val_dataloader:
        with torch.no_grad():
            self.model.validation_step(self.prepare_batch(batch))
        self.val_batch_idx += 1

In [10]:
model = LinearRegression(2, lr=0.03)
data = d2l.SyntheticRegressionData(w=torch.tensor([2, -3.4]), b=4.2)
trainer = d2l.Trainer(max_epochs=3)
trainer.fit(model, data)

AssertionError: Neural network is defined

In [11]:
with torch.no_grad():
    print(f'error in estimating w: {data.w - model.w.reshape(data.w.shape)}') #data.w = real value, model.w = param model learned, 
    print(f'error in estimating b: {data.b - model.b}')

error in estimating w: tensor([ 1.9896, -3.4005])
error in estimating b: tensor([4.2000])


On branch main
Your branch is ahead of 'origin/main' by 1 commit.
  (use "git push" to publish your local commits)

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   ../Chapter 2/prob-and-stats.ipynb

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	../.DS_Store

no changes added to commit (use "git add" and/or "git commit -a")
Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 11 threads
Compressing objects: 100% (6/6), done.
Writing objects: 100% (6/6), 4.94 KiB | 4.94 MiB/s, done.
Total 6 (delta 1), reused 0 (delta 0), pack-reused 0 (from 0)
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
To https://github.com/anish-band/d2l-notebooks.git
   bb23051..880ad39  main -> main
